In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt

project_dir = "/home/S22/workspace/BLAST"
vector_dir = project_dir + '/feature_construction/output/'
save_dir = project_dir + '/dimension_reduction/grid_mapping/'
save_scaled_reduced_data_dir = save_dir + 'scaled_reduced_data/'
os.makedirs(save_dir, exist_ok=True)
os.makedirs(save_scaled_reduced_data_dir, exist_ok=True) # 存储降维之后数据的目录

RESOLUTION = 0.01
SAMPLING_RATE = 0.1
GRID_SIZE = int(1 / RESOLUTION)
MARK = f"0{len(str(GRID_SIZE))}"

def scatter_plot(data, title, save_name=None):
    # 降维可视化
    plt.scatter(data[:, 0], data[:, 1], s=5)
    plt.title(title)
    if save_name:
        # 保存热力图
        save_path = save_dir + save_name + ".png"
        plt.savefig(save_dir + save_name + ".png")
        print(f"Scatter saved to {save_path}")
    plt.show()


In [4]:
# 按照数据比例在不同的数据集中等比例抽样
from tqdm import tqdm
current_index = 0
dataset_range = {}
all_data = []
for dataset_part in tqdm(os.listdir(vector_dir)):
    data = np.load(os.path.join(vector_dir, dataset_part)).astype(np.float32)
    n, c, l = data.shape
    dataset_range[dataset_part] = (current_index, current_index + n * c)
    all_data.append(data.reshape(n * c, l))
    current_index += n * c
all_data_np = np.concatenate(all_data, axis=0)
del all_data


100%|██████████| 755/755 [00:04<00:00, 183.39it/s]


In [3]:
# 采样
NUM_SAMPLES = int(all_data_np.shape[0] * SAMPLING_RATE) # 原始数据4100w左右，采样2%，大概80w
sample_idx = np.random.choice(all_data_np.shape[0], NUM_SAMPLES, replace=False)
sample_data = all_data_np[sample_idx]

print(f"sampling {NUM_SAMPLES} data from {all_data_np.shape[0]} data, {NUM_SAMPLES/all_data_np.shape[0]*100:.2f}%")

del all_data_np


sampling 4180574 data from 41805743 data, 10.00%


In [4]:
import umap
from sklearn.preprocessing import MinMaxScaler

umap_args = {'n_components': 2, 'n_neighbors': 100, 'min_dist': 0.9, 'metric': 'hamming', 'n_jobs': -1, 'init': 'random', 'verbose': True, 'low_memory': False}

suffix = '_'.join([f"{key}={value}" for key, value in umap_args.items()])
suffix += f"_sample_rate={SAMPLING_RATE}"

model = umap.UMAP(**umap_args)
scaler = MinMaxScaler()

print("Dimension reduction...")
reduced_data = model.fit_transform(sample_data)
normalized_data = scaler.fit_transform(reduced_data)
np.save(save_dir + f"normalized_data_{suffix}.npy", normalized_data)


Dimension reduction...
UMAP(init='random', low_memory=False, metric='hamming', min_dist=0.9, n_neighbors=100, verbose=True)


/home/S22/anaconda3/envs/th231/lib/python3.11/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(


Thu Jan 16 16:59:24 2025 Construct fuzzy simplicial set
Thu Jan 16 16:59:25 2025 Finding Nearest Neighbors
Thu Jan 16 16:59:25 2025 Building RP forest with 64 trees
Thu Jan 16 16:59:49 2025 NN descent for 22 iterations
	 1  /  22
	 2  /  22
	Stopping threshold met -- exiting after 2 iterations
Thu Jan 16 18:24:58 2025 Finished Nearest Neighbor Search
Thu Jan 16 18:28:53 2025 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Thu Jan 16 18:48:09 2025 Finished embedding


In [5]:
# save model
import pickle
model_path = save_dir + f"umap_model_{suffix}.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
    print(f"Model saved to {model_path}")
# save scaler
scaler_path = save_dir + f"scaler_{suffix}.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
    print(f"Scaler saved to {scaler_path}")

Thu Jan 16 19:18:15 2025 Worst tree score: 0.48501402
Thu Jan 16 19:18:15 2025 Mean tree score: 0.54496601
Thu Jan 16 19:18:15 2025 Best tree score: 0.92273023
Thu Jan 16 19:19:04 2025 Forward diversification reduced edges from 418057400 to 384171196
Thu Jan 16 19:19:19 2025 Reverse diversification reduced edges from 384171196 to 381610329


/home/S22/anaconda3/envs/th231/lib/python3.11/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


Thu Jan 16 19:20:35 2025 Degree pruning reduced edges from 439034632 to 439020570
Thu Jan 16 19:20:35 2025 Resorting data and graph based on tree order
Thu Jan 16 19:20:55 2025 Building and compiling search function
Model saved to /home/S22/workspace/Datasets/time_series_analysis/grid_mapping/umap_model_n_components=2_n_neighbors=100_min_dist=0.9_metric=hamming_n_jobs=-1_init=random_verbose=True_low_memory=False_sample_rate=0.1.pkl
Scaler saved to /home/S22/workspace/Datasets/time_series_analysis/grid_mapping/scaler_n_components=2_n_neighbors=100_min_dist=0.9_metric=hamming_n_jobs=-1_init=random_verbose=True_low_memory=False_sample_rate=0.1.pkl


In [ ]:
import json
from joblib import Parallel, delayed
from datetime import datetime

# 遍历数据集Part的向量，并把降维之后的数据写入栅格中
def init_grid():
    grid = {}
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            key = f"{i:{MARK}d}__{j:{MARK}d}"
            grid[key] = []
    return grid

def process_dataset_part_vector(dataset_part_path, model, scaler):
    # 读取数据并降维
    dataset_part_vector = np.load(dataset_part_path)
    n, c, l = dataset_part_vector.shape
    dataset_part_vector = dataset_part_vector.reshape(n*c, l)
    reduced_data = model.transform(dataset_part_vector)
    scaled_reduced_data = scaler.transform(reduced_data)
    scaled_reduced_data = scaled_reduced_data.reshape(n, c, 2)
    save_path = save_scaled_reduced_data_dir + dataset_part_path.split("/")[-1]
    np.save(save_path, scaled_reduced_data)

    # 构建该数据集Part的栅格
    # 初始化
    dataset_part_grid = init_grid()
    # 统计grid中的数据
    for i in range(n):
        for j in range(c):
            x, y = scaled_reduced_data[i, j] # 降维之后的坐标
            x_idx = int(x * (GRID_SIZE - 1)) # x栅格坐标
            y_idx = int(y * (GRID_SIZE - 1)) # y栅格坐标
            # 处理超出边界的情况
            if x_idx >= GRID_SIZE: x_idx = GRID_SIZE - 1
            if y_idx >= GRID_SIZE: y_idx = GRID_SIZE - 1
            if x_idx < 0: x_idx = 0
            if y_idx < 0: y_idx = 0
            key = f"{x_idx:{MARK}d}__{y_idx:{MARK}d}"
            dataset_part_name = dataset_part_path.split("/")[-1].split(".npy")[0]
            value = f"{dataset_part_name}_{i}_{j}"
            dataset_part_grid[key].append(value)

    return dataset_part_grid


# 顺序读取所有的数据集Part向量
# 单线程即可，因为UMap是多线程的，多线程+多线程会导致开销很大，反而不如单线程
from tqdm import tqdm
dataset_part_grids = []
for dataset_part_vector_path in tqdm(os.listdir(vector_dir)):
    dataset_part_grid = process_dataset_part_vector(os.path.join(vector_dir, dataset_part_vector_path), model, scaler)
    dataset_part_grids.append(dataset_part_grid)

# 合并字典字典
from tqdm import tqdm
grid_mapping = init_grid()
for dataset_part_grid in tqdm(dataset_part_grids):
    for key, value in dataset_part_grid.items():
        grid_mapping[key].extend(value)

# 保存grid_mapping
grid_mapping_path = os.path.join(save_dir, "grid.json")
with open(grid_mapping_path, "w") as f:
    print(f"Grid mapping saved to {grid_mapping_path}")
    json.dump(grid_mapping, f)
